In [1]:
import pandas as pd
import chardet

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
#data = pd.read_excel("drive/MyDrive/preprocessed_lenta_titles.xlsx")
#data = data.drop(columns=['text_lower','text_punct','text_stop','text_common','text_rare','text_nonum','text_token','text_lemm','text_ready'])
data24 = data[data['tags']=='Политика']
d3 = len(data24)
data24.head()
print(d3)

16506


In [13]:
!pip3 install urllib3==1.25.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.5/125.5 kB 11.5 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.2.3
    Uninstalling urllib3-2.2.3:
      Successfully uninstalled urllib3-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentry-sdk 2.19.0 requires urllib3>=1.26.11, but you have urllib3 1.25.4 which is incompatible.


In [14]:
#!wget https://raw.githubusercontent.com/sberbank-ai/ru-gpts/master/pretrain_transformers.py

In [ ]:
#!wget https://raw.githubusercontent.com/sberbank-ai/ru-gpts/master/generate_transformers.py

--2024-10-14 06:49:38--  https://raw.githubusercontent.com/sberbank-ai/ru-gpts/master/generate_transformers.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10474 (10K) [text/plain]
Saving to: ‘generate_transformers.py’

generate_transforme 100%[===================>]  10.23K  --.-KB/s    in 0s      

2024-10-14 06:49:38 (113 MB/s) - ‘generate_transformers.py’ saved [10474/10474]



In [15]:
%%writefile setup.sh
git clone https://github.com/NVIDIA/apex
cd apex
# if pip >= 23.1 (ref: https://pip.pypa.io/en/stable/news/#v23-1) which supports multiple `--config-settings` with the same key...
pip install -v --disable-pip-version-check --no-cache-dir --no-build-isolation --config-settings "--build-option=--cpp_ext" --config-settings "--build-option=--cuda_ext" ./
# otherwise
pip install -v --disable-pip-version-check --no-cache-dir --no-build-isolation --global-option="--cpp_ext" --global-option="--cuda_ext" ./

Writing setup.sh


In [16]:
!sh setup.sh

Cloning into 'apex'...
remote: Enumerating objects: 11927, done.
remote: Counting objects: 100% (3995/3995), done.
remote: Compressing objects: 100% (769/769), done.
remote: Total 11927 (delta 3513), reused 3436 (delta 3219), pack-reused 7932 (from 1)
Receiving objects: 100% (11927/11927), 15.61 MiB | 17.17 MiB/s, done.
Resolving deltas: 100% (8342/8342), done.
Using pip 24.1.2 from /usr/local/lib/python3.10/dist-packages/pip (python 3.10)
Processing /content/apex
  Running command Preparing metadata (pyproject.toml)


  torch.__version__  = 2.5.1+cu121


  running dist_info
  creating /tmp/pip-modern-metadata-aj4r_spi/apex.egg-info
  writing /tmp/pip-modern-metadata-aj4r_spi/apex.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-modern-metadata-aj4r_spi/apex.egg-info/dependency_links.txt
  writing requirements to /tmp/pip-modern-metadata-aj4r_spi/apex.egg-info/requires.txt
  writing top-level names to /tmp/pip-modern-metadata-aj4r_spi/apex.egg-info/top_level.txt
  writing manif

In [17]:
import numpy as np
import random

In [18]:
random.seed(4673)
np.random.seed(4673)

In [19]:
val_ind = random.sample(range(data.shape[0]), 5000)

In [20]:
train = [data24.iloc[i]['title'] for i in range(len(data24)) if i not in val_ind]
valid = [data24.iloc[i]['title'] for i in range(len(data24)) if i in val_ind]

In [21]:
len(train), len(valid)

(15338, 1168)

In [22]:
with open("train.txt", "w") as file:
    file.write("\n".join(train))

In [23]:
with open("valid.txt", "w") as file:
    file.write("\n".join(valid))

In [24]:
!python drive/MyDrive/pretrain_transformers.py \
    --output_dir=drive/MyDrive/pretrain \
    --model_type=gpt2 \
    --model_name_or_path=sberbank-ai/rugpt3small_based_on_gpt2 \
    --do_train \
    --train_data_file=train.txt \
    --do_eval \
    --fp16 \
    --eval_data_file=valid.txt \
    --per_gpu_train_batch_size 64 \
    --gradient_accumulation_steps 1 \
    --num_train_epochs 2 \
    --block_size 64 \
    --overwrite_output_dir

2024-12-07 08:04:21.195779: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-07 08:04:21.228375: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-07 08:04:21.238404: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-07 08:04:21.261121: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-07 08:04:24.586537: W tensorflow/comp

In [25]:
!python drive/MyDrive/generate_transformers.py \
    --model_type=gpt2 \
    --model_name_or_path=drive/MyDrive/pretrain \
    --k=6 \
    --p=1.4 \
    --length=100

2024-12-07 08:06:26.048385: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-07 08:06:26.080690: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-07 08:06:26.090790: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-07 08:06:26.114073: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-12-07 08:06:27.605813: W tensorflow/comp